# 04: Feature Representation and Baselines

This notebook builds the shared per-type encoders and the raw/classical baselines, applied identically to whichever candidate is active, using new code in `src/features/` alongside reused utilities from `src/eda/`.

## Setup

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import config

PROCESSED_DIR = config.PROCESSED_DIR
FINAL_DIR = config.FINAL_DIR
FINAL_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

## 1. Load active candidate

In [ ]:
# Which candidate is active is set once, in src/config.py's
# ACTIVE_CANDIDATE -- change it there, not here, so every downstream cell
# (and every module in src/features/) picks up the same value without
# this notebook ever naming a candidate itself.
ACTIVE_CANDIDATE = config.ACTIVE_CANDIDATE

CANDIDATE_PATHS = {
    split: FINAL_DIR / f"tmdb_{ACTIVE_CANDIDATE}_{split}.parquet"
    for split in ("train", "val", "test")
}

In [ ]:
train_df = pd.read_parquet(CANDIDATE_PATHS["train"])
val_df = pd.read_parquet(CANDIDATE_PATHS["val"])
test_df = pd.read_parquet(CANDIDATE_PATHS["test"])

print(f"Active candidate: {ACTIVE_CANDIDATE}")
print(f"  train: {train_df.shape}")
print(f"  val:   {val_df.shape}")
print(f"  test:  {test_df.shape}") 

In [ ]:
train_df.dtypes

## 2. Target/feature split

In [ ]:
from src.features.target_split import build_feature_target_split

train_split = build_feature_target_split(train_df)

print(f"target_col: {train_split.target_col}")
print(f"id_cols: {len(train_split.id_cols)}")
print(f"feature_cols: {len(train_split.feature_cols)}")
print(f"first 5 feature_cols: {train_split.feature_cols}")

In [ ]:
val_split = build_feature_target_split(val_df)
test_split = build_feature_target_split(test_df)

assert train_split.feature_cols == val_split.feature_cols == test_split.feature_cols, (
    "feature_cols differ across splits: "
    f"train={train_split.feature_cols}, val={val_split.feature_cols}, test={test_split.feature_cols}"
)
assert train_split.id_cols == val_split.id_cols == test_split.id_cols, (
    "id_cols differ across splits: "
    f"train={train_split.id_cols}, val={val_split.id_cols}, test={test_split.id_cols}"
)

print("Feature/target split consistent across train/val/test")

### Column groups (derived from actual data, not a hardcoded list)

In [ ]:
from src.features.target_split import build_column_groups

column_groups = build_column_groups(train_df, train_split)

print(f"numeric_cols: {column_groups.numeric_cols}")
print(f"bypass_cols: {column_groups.bypass_cols}")
print(f"categorical_cols: {column_groups.categorical_cols}")
print(f"text_cols: {column_groups.text_cols}")
print(f"list_cols: {column_groups.list_cols}")

## 3. Numeric standardisation

In [ ]:
NUMERIC_COLS = column_groups.numeric_cols
print(NUMERIC_COLS)

Matches: 
- {budget, revenue, runtime, popularity, release_year} for br_gt0
- {runtime, popularity, release_year} for nbr_gt5 (budget/revenue absent)

### Fit on train, transform all three splits

In [ ]:
from src.features.standardize import Standardizer

std = Standardizer(NUMERIC_COLS)
std.fit(train_df)

train_df_std = std.transform(train_df)
val_df_std = std.transform(val_df)
test_df_std = std.transform(test_df)

print("train mean:")
print(train_df_std[NUMERIC_COLS].mean())
print("train std:")
print(train_df_std[NUMERIC_COLS].std())

print("val mean:")
print(val_df_std[NUMERIC_COLS].mean())
print("val std:")
print(val_df_std[NUMERIC_COLS].std())

### Unit check: manual z-score vs. `Standardizer` output

In [ ]:
manual_runtime_std = (train_df["runtime"] - train_df["runtime"].mean()) / train_df["runtime"].std(ddof=0)
class_runtime_std = train_df_std["runtime"]

print(f"Exact match: {manual_runtime_std.equals(class_runtime_std)}")
print(f"Max absolute difference: {(manual_runtime_std - class_runtime_std).abs().max()}")

### Persist the fitted `Standardizer`

In [ ]:
from src.features.helper import save_json, load_json, save_representation_with_id

ENCODER_DIR = config.ENCODER_DIR

standardizer_path = save_json(std, "standardizer.json", ENCODER_DIR)
std_reloaded = load_json(Standardizer, standardizer_path)

assert std_reloaded.mean_.equals(std.mean_), "reloaded mean_ does not match the original"
assert std_reloaded.std_.equals(std.std_), "reloaded std_ does not match the original"
print("Round-trip check passed: reloaded mean_/std_ match the original exactly")

## 4. Piecewise-linear encoding (PLE)

`release_month_sin` and `release_month_cos` bypass the PLE module as fixed cases of Eq. 3.3, with `c = 1/12` and `k = 1`. They should be adjacent in Step 9’s output without needing `np.concatenate`. The learned-frequency form of Eq. 3.3 isn't used elsewhere.

Fits on the *standardised* numeric columns (Eq. 3.1 -> Eq. 3.2 ordering: standardise first, then bin the standardised values, not the raw ones).

In [ ]:
from src.features.numeric_embeddings import PiecewiseLinearEncoder

ple = PiecewiseLinearEncoder(NUMERIC_COLS, n_bins=32)
ple.fit(train_df_std)

train_ple = ple.transform(train_df_std)
val_ple = ple.transform(val_df_std)
test_ple = ple.transform(test_df_std)

for col in NUMERIC_COLS:
    n_bins_used = ple.n_bins_[col]
    trivial_removed = n_bins_used < ple.n_bins
    print(
        f"{col}: train {train_ple[col].shape}, val {val_ple[col].shape}, "
        f"test {test_ple[col].shape} (n_bins_used={n_bins_used}"
        f"{', trivial bins removed' if trivial_removed else ''})"
    )

### Verify PLE against a hand-computed value

In [ ]:
PLE_TEST_COL = "runtime"

ple_test = PiecewiseLinearEncoder([PLE_TEST_COL], n_bins=4)
ple_test.fit(train_df_std)
edges = ple_test.edges_[PLE_TEST_COL]
assert ple_test.n_bins_[PLE_TEST_COL] == 4, "expected no trivial bins for this test"

# A value known to sit inside the 2nd bin [b1, b2): the midpoint.
x_bin2 = (edges[1] + edges[2]) / 2
expected = np.array([1.0, 0.5, 0.0, 0.0])
actual = ple_test.transform(pd.DataFrame({PLE_TEST_COL: [x_bin2]}))[PLE_TEST_COL][0]

print(f"x = {x_bin2:.4f} (bin 2 of edges {edges})")
print(f"expected: {expected}")
print(f"actual:   {actual}")
assert np.allclose(actual, expected), "PLE output does not match the hand-computed expectation"

# A value beyond the training max should still produce a valid, unclipped output.
x_large = train_df_std[PLE_TEST_COL].max() + 1000
out_large = ple_test.transform(pd.DataFrame({PLE_TEST_COL: [x_large]}))[PLE_TEST_COL][0]
print(f"x = {x_large:.2f} (beyond train max): {out_large}")
assert np.isfinite(out_large).all(), "out-of-range value produced NaN/inf"
assert out_large[-1] >= 1, "e_T should be >= 1 for a value beyond the training max"

print("Verification passed: PLE matches the hand-computed value and handles out-of-range input safely.")

### Persist the fitted PLE encoder

In [ ]:
ple_path = save_json(ple, "ple.json", ENCODER_DIR)
ple_reloaded = load_json(PiecewiseLinearEncoder, ple_path)

for col in ple.cols:
    assert np.array_equal(ple_reloaded.edges_[col], ple.edges_[col]), (
        f"reloaded edges_ for {col} do not match the original"
    )

val_ple_reloaded = ple_reloaded.transform(val_df_std)
for col in ple.cols:
    assert np.array_equal(val_ple_reloaded[col], val_ple[col]), (
        f"reloaded transform() for {col} does not match the original"
    )

print("Round-trip check passed: reloaded bin edges and val transform match the original exactly")

## 5. Categorical features

In [ ]:
print(train_df[train_split.feature_cols].select_dtypes(include=["category"]).columns.tolist())

`original_language` is the only scalar categorical remaining, matching status/adult having already been dropped in 01/02.

### Fit `CategoricalLookup` on `original_language`

In [ ]:
from src.features.categorical import CategoricalLookup

assert len(column_groups.categorical_cols) == 1, (
    f"expected exactly one categorical column, got {column_groups.categorical_cols}"
)
lang_col = column_groups.categorical_cols[0]

categorical_lookup = CategoricalLookup(lang_col)
categorical_lookup.fit(train_df)

train_lang_idx = categorical_lookup.transform(train_df)
val_lang_idx = categorical_lookup.transform(val_df)
test_lang_idx = categorical_lookup.transform(test_df)

vocab_size = len(categorical_lookup.token_to_index)
print(f"vocab size (excluding reserved index 0): {vocab_size}")

assert vocab_size == config.EXPECTED_VOCAB_SIZE[ACTIVE_CANDIDATE], (
    f"vocab size {vocab_size} does not match 03_feature_split.ipynb's "
    f"reported {config.EXPECTED_VOCAB_SIZE[ACTIVE_CANDIDATE]} for {ACTIVE_CANDIDATE}"
)
print(f"Matches 03_feature_split.ipynb's reported cardinality for {ACTIVE_CANDIDATE}")

n_unseen_val = int((val_lang_idx == 0).sum())
n_unseen_test = int((test_lang_idx == 0).sum())
print(f"val rows hitting the unseen-token fallback: {n_unseen_val}")
print(f"test rows hitting the unseen-token fallback: {n_unseen_test}")

### Persist the fitted `CategoricalLookup`

In [ ]:
categorical_lookup_path = save_json(
    categorical_lookup, "categorical_lookup_original_language.json", ENCODER_DIR
)
categorical_lookup_reloaded = load_json(CategoricalLookup, categorical_lookup_path)

assert categorical_lookup_reloaded.token_to_index == categorical_lookup.token_to_index, (
    "reloaded token_to_index does not match the original"
)

val_lang_idx_reloaded = categorical_lookup_reloaded.transform(val_df)
assert np.array_equal(val_lang_idx_reloaded, val_lang_idx), (
    "reloaded transform() does not match the original"
)

print("Round-trip check passed: reloaded token_to_index and val transform match the original exactly")

## 6. Text encoding (`overview`, `original_title`)

`paraphrase-multilingual-mpnet-base-v2` (`config.SBERT_MODEL_NAME`) is confirmed as final for text encoding. \
Reason: A monolingual English encoder would confound "language recoverable from the embedding" with "language recoverable because the encoder can't represent non-English text well in the first place"

In [ ]:
import time

from src.features.text_encoder import TextEncoder

TEXT_FIELDS = column_groups.text_cols
SPLITS = {"train": train_df, "val": val_df, "test": test_df}

# Loaded once and reused for every field/split -- the model itself is
# expensive to load, not the encoding calls, so a fresh TextEncoder per
# field would waste that cost five times over.
text_encoder = TextEncoder()

ENCODERS_DIR = ENCODER_DIR
ENCODERS_DIR.mkdir(parents=True, exist_ok=True)

embedding_cache_status = []

for field in TEXT_FIELDS:
    field_start = time.time()
    for split_name, df in SPLITS.items():
        out_path = ENCODERS_DIR / f"{field}_embeddings_{split_name}.npy"
        if not out_path.exists():
            embeddings = text_encoder.encode(df[field].tolist())
            np.save(out_path, embeddings)
            status = "encoded"
        else:
            embeddings = np.load(out_path)
            status = "cached"
        print(f"{field} / {split_name}: {embeddings.shape} ({status})")
        embedding_cache_status.append(
            {"field": field, "split": split_name, "shape": embeddings.shape, "status": status}
        )
    field_elapsed = time.time() - field_start
    print(f"{field}: {field_elapsed:.1f}s for all three splits\n")

embedding_cache_summary = pd.DataFrame(embedding_cache_status)
embedding_cache_summary

## 7. List-valued field cardinality check

The five list-valued fields: `genres`, `keywords`, `production_companies`, `production_countries`, `spoken_languages`.

In [ ]:
LIST_VALUED_FIELDS = column_groups.list_cols

for col in LIST_VALUED_FIELDS:
    print(f"{col}: {train_df[col].explode().nunique()} unique values")

### Sanity-check `ListFieldPooler` on `genres` alone, before looping

In [ ]:
from src.features.list_pooling import ListFieldPooler

def oov_pct(idx_series):
    all_indices = [i for row in idx_series for i in row]
    if not all_indices:
        return 0.0
    return sum(1 for i in all_indices if i == 0) / len(all_indices) * 100


genres_pooler = ListFieldPooler("genres", embedding_dim=8)
genres_pooler.fit(train_df)

train_genres_idx = genres_pooler.transform(train_df)
val_genres_idx = genres_pooler.transform(val_df)
test_genres_idx = genres_pooler.transform(test_df)

genres_vocab_size = len(genres_pooler.token_to_index)
genres_avg_list_len = train_genres_idx.apply(len).mean()
genres_oov_val = oov_pct(val_genres_idx)
genres_oov_test = oov_pct(test_genres_idx)

print(f"genres: vocab_size={genres_vocab_size}, avg list length (train)={genres_avg_list_len:.2f}")
print(f"genres: OOV% val={genres_oov_val:.2f}%, OOV% test={genres_oov_test:.2f}%")

### Loop `ListFieldPooler` over all five list-valued fields

In [ ]:
list_poolers = {}
list_pooling_summary = []

for field in LIST_VALUED_FIELDS:
    pooler = ListFieldPooler(field, embedding_dim=8)
    pooler.fit(train_df)
    list_poolers[field] = pooler

    train_idx = pooler.transform(train_df)
    val_idx = pooler.transform(val_df)
    test_idx = pooler.transform(test_df)

    list_pooling_summary.append(
        {
            "field": field,
            "vocab_size": len(pooler.token_to_index),
            "avg_list_length": round(train_idx.apply(len).mean(), 2),
            "oov_pct_val": round(oov_pct(val_idx), 2),
            "oov_pct_test": round(oov_pct(test_idx), 2),
        }
    )

list_pooling_summary_df = pd.DataFrame(list_pooling_summary)
list_pooling_summary_df

### Persist the fitted `ListFieldPooler`

In [ ]:
list_pooler_paths = {}
for field in LIST_VALUED_FIELDS:
    list_pooler_paths[field] = save_json(
        list_poolers[field], f"list_pooler_{field}.json", ENCODER_DIR
    )

for field in LIST_VALUED_FIELDS:
    reloaded = load_json(ListFieldPooler, list_pooler_paths[field])
    assert reloaded.token_to_index == list_poolers[field].token_to_index, (
        f"reloaded token_to_index for {field} does not match the original"
    )

print("Round-trip check passed: all five reloaded poolers match the originals exactly")

## 8. Raw and classical baselines

### `build_raw_baseline`

In [ ]:
from src.features.baselines import build_raw_baseline

train_raw_baseline = build_raw_baseline(
    train_df, train_split, NUMERIC_COLS, categorical_lookup, LIST_VALUED_FIELDS
)
print(f"raw baseline shape: {train_raw_baseline.shape}")

non_numeric_cols = [
    c for c in train_raw_baseline.columns if not pd.api.types.is_numeric_dtype(train_raw_baseline[c])
]
assert not non_numeric_cols, f"raw baseline has non-numeric columns: {non_numeric_cols}"
print("All columns confirmed numeric dtype")

### `build_classical_baseline`

In [ ]:
from src.features.baselines import build_classical_baseline, EXCLUDED_TEXT_COLS

train_classical_baseline = build_classical_baseline(
    train_df, train_split, NUMERIC_COLS, std, categorical_lookup, list_poolers
)
print(f"classical baseline shape: {train_classical_baseline.shape}")

# Explicit arithmetic, not eyeballed: numeric cols + one-hot width
# (real language tokens + OOV) + multi-hot width per list field
# (real vocab items + OOV, summed across fields) + whatever's left
# in feature_cols after accounting for numeric/language/list/text.
one_hot_width = len(categorical_lookup.token_to_index) + 1
multi_hot_width = sum(len(pooler.token_to_index) + 1 for pooler in list_poolers.values())
n_passthrough = (
    len(train_split.feature_cols)
    - len(NUMERIC_COLS)
    - 1  # original_language
    - len(list_poolers)
    - len(EXCLUDED_TEXT_COLS)
)
expected_classical_cols = len(NUMERIC_COLS) + one_hot_width + multi_hot_width + n_passthrough

print(
    f"expected = {len(NUMERIC_COLS)} numeric + {one_hot_width} one-hot "
    f"+ {multi_hot_width} multi-hot + {n_passthrough} passthrough = {expected_classical_cols}"
)
assert train_classical_baseline.shape[1] == expected_classical_cols, (
    f"classical baseline column count {train_classical_baseline.shape[1]} != "
    f"expected {expected_classical_cols}"
)
print("Column count matches the explicit arithmetic")

### Compare shapes

In [ ]:
print(f"raw baseline shape: {train_raw_baseline.shape}")
print(f"classical baseline shape: {train_classical_baseline.shape}")

col_diff = train_classical_baseline.shape[1] - train_raw_baseline.shape[1]
print(f"column count difference: {train_classical_baseline.shape[1]} - {train_raw_baseline.shape[1]} = {col_diff}")

MIN_EXPECTED_DIFF = 10
if col_diff < MIN_EXPECTED_DIFF:
    print(
        f"WARNING: raw ({train_raw_baseline.shape[1]}) and classical "
        f"({train_classical_baseline.shape[1]}) column counts are suspiciously "
        "close -- double-check rather than assume this is fine"
    )
else:
    print(
        "Column counts differ substantially, as expected: raw has one "
        "column per numeric feature + one label-encoded language column + "
        "one count column per list field, while classical expands "
        "language to one-hot width and each list field to multi-hot width"
    )

### Save raw and classical baselines for all three splits

Loaded directly by Step 9/Section 3.5's evaluation code -- built fresh per split here (train/val/test alike) using the same already-fitted `std`/`categorical_lookup`/`list_poolers`, never refit.

In [ ]:
RAW_MODEL_DIR = config.MODELS_DIR / "raw"
CLASSICAL_MODEL_DIR = config.MODELS_DIR / "classical"
RAW_MODEL_DIR.mkdir(parents=True, exist_ok=True)
CLASSICAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_SPLITS = {"train": train_df, "val": val_df, "test": test_df}

for split_name, df in FINAL_SPLITS.items():
    raw_baseline = build_raw_baseline(
        df, train_split, NUMERIC_COLS, categorical_lookup, LIST_VALUED_FIELDS
    )
    raw_path = RAW_MODEL_DIR / f"{split_name}.parquet"
    save_representation_with_id(raw_baseline, df["id"], raw_path)

    classical_baseline = build_classical_baseline(
        df, train_split, NUMERIC_COLS, std, categorical_lookup, list_poolers
    )
    classical_path = CLASSICAL_MODEL_DIR / f"{split_name}.parquet"
    save_representation_with_id(classical_baseline, df["id"], classical_path)

## 9. Fitted encoder bundle for Embedding Paradigms

`FittedEncoders` in `src/features/pipeline.py` includes fitted encoders (Step 3 to 8), including cached SBERT vectors from `ENCODER_DIR/`.

**Why the SBERT cache is keyed by `id`, not row position.** SCARF calls `transform()` twice per batch, necessitating offline encoding in Step 6 to avoid slowdowns. Only the `id` column survives corruption, ensuring correct identification. Missing ids raise errors, as live encoding isn't permitted. All rows must be prepared by Step 6.

In [ ]:
from src.features.pipeline import FittedEncoders

fitted_encoders = FittedEncoders.fit(FINAL_SPLITS, train_split, ENCODER_DIR)

print(f"numeric_cols: {fitted_encoders.column_groups.numeric_cols}")
print(f"bypass_cols: {fitted_encoders.column_groups.bypass_cols}")
print(f"list_poolers: {sorted(fitted_encoders.list_poolers.keys())}")
print(f"categorical_cols: {fitted_encoders.column_groups.categorical_cols}")
for field in fitted_encoders.text_cache:
    print(f"text_cache[{field!r}]: {len(fitted_encoders.text_cache[field])} ids cached")

### Dimensions for sizing paradigms

`fitted_encoders.dims` provides one number per `transform()` key for Table 3.2, which defines feature projections and input width. The values vary: `"numeric"` is an array width, `"overview"` and `"original_title"` are vector dimensions, while `"original_language"` and list fields indicate vocabulary sizes for `nn.Embedding` tables.

In [ ]:
print(fitted_encoders.dims)

### Verify `transform()` on train_df

In [ ]:
train_transformed = fitted_encoders.transform(train_df)
for key, arr in train_transformed.items():
    print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

### Verify `include_text=False` omits TEXT_FIELDS entirely

`include_text` defaults to `True`, so every existing call site above is unaffected. Passing `include_text=False` must drop `"overview"`/`"original_title"` from the result dict entirely -- while leaving every other key untouched; passing `include_text=True` explicitly must match the default call exactly.

In [ ]:
from src.features.pipeline import TEXT_FIELDS

train_transformed_no_text = fitted_encoders.transform(train_df, include_text=False)
train_transformed_explicit_true = fitted_encoders.transform(train_df, include_text=True)

present_text_keys = [f for f in TEXT_FIELDS if f in train_transformed_no_text]
assert not present_text_keys, (
    f"include_text=False should omit {TEXT_FIELDS} entirely, but found: {present_text_keys}"
)

non_text_keys = set(train_transformed.keys()) - set(TEXT_FIELDS)
assert set(train_transformed_no_text.keys()) == non_text_keys, (
    f"include_text=False changed the non-text keys: "
    f"{set(train_transformed_no_text.keys())} != {non_text_keys}"
)

assert set(train_transformed_explicit_true.keys()) == set(train_transformed.keys()), (
    "include_text=True (explicit) does not match the default (omitted) key set"
)
for key, arr_default in train_transformed.items():
    arr_explicit = train_transformed_explicit_true[key]
    if arr_default.dtype == object:
        matches = all(list(a) == list(b) for a, b in zip(arr_default, arr_explicit))
    else:
        matches = np.array_equal(arr_default, arr_explicit)
    assert matches, f"{key} differs between the default call and explicit include_text=True"

print(f"include_text=False omits {TEXT_FIELDS} entirely: PASSED")
print(f"non-text keys unchanged: {sorted(non_text_keys)}")
print("include_text=True (explicit) matches the default (omitted) call exactly: PASSED")

### Verify the SBERT lookup is id-based, not positional

A shuffled copy of `train_df` must still resolve each row's SBERT vectors to the *same* id, not to whatever row happens to sit at that position

In [ ]:
shuffled_train_df = train_df.sample(frac=1.0, random_state=config.RANDOM_SEED).reset_index(drop=True)
shuffled_transformed = fitted_encoders.transform(shuffled_train_df)

mismatches = 0
for shuffled_pos, row_id in enumerate(shuffled_train_df["id"]):
    original_pos = train_df.index[train_df["id"] == row_id][0]
    if not np.array_equal(
        shuffled_transformed["overview"][shuffled_pos],
        train_transformed["overview"][original_pos],
    ):
        mismatches += 1

assert mismatches == 0, f"{mismatches} rows resolved to the wrong SBERT vector after shuffling"
print("Shuffled-row SBERT lookup matches by id for every row: PASSED")

### Verify `save()`/`load()` round-trip

In [ ]:
fitted_encoders.save(ENCODER_DIR)
reloaded_encoders = FittedEncoders.fit(FINAL_SPLITS, train_split, ENCODER_DIR)

assert reloaded_encoders.dims == fitted_encoders.dims, "dims differ after save/load"
assert reloaded_encoders.corruption_eligible_cols == fitted_encoders.corruption_eligible_cols, (
    "corruption_eligible_cols differ after save/load"
)

val_transformed_before = fitted_encoders.transform(val_df)
val_transformed_after = reloaded_encoders.transform(val_df)
assert set(val_transformed_before.keys()) == set(val_transformed_after.keys())
for key, arr_before in val_transformed_before.items():
    arr_after = val_transformed_after[key]
    if arr_before.dtype == object:
        matches = all(list(a) == list(b) for a, b in zip(arr_before, arr_after))
    else:
        matches = np.array_equal(arr_before, arr_after)
    assert matches, f"{key} differs after save/load"

print("save() -> fit() round-trip: dims, corruption_eligible_cols, and "
      "transform(val_df) all identical to the pre-save instance")

### Confirm `transform()` is cheap on every call

The SCARF model's loop calls `transform()` twice per batch, every batch -- so it must be a pure in-memory lookup with no re-fitting and no per-call disk reads, and its cost must not depend on which rows or which columns happen to be corrupted. Timing three calls -- `train_df` twice, then a copy with a few numeric/categorical columns corrupted (mimicking SCARF's own corruption step) -- checks this directly: if the corrupted-df call is meaningfully slower, something is doing unnecessary per-call work (e.g. re-deriving the SBERT cache key per row) that needs fixing before this goes anywhere near training.

In [ ]:
import time

def timed_transform(df, n_reps=5):
    times = []
    for _ in range(n_reps):
        start = time.perf_counter()
        fitted_encoders.transform(df)
        times.append(time.perf_counter() - start)
    return min(times)  # min is the most stable signal against transient OS/GC noise

call1_time = timed_transform(train_df)
call2_time = timed_transform(train_df)

# Mimics SCARF-style corruption: swap a few numeric/categorical columns'
# values across rows. id and both text fields are left untouched, matching
# corruption_eligible_cols' default exclusion of "overview"/"original_title".
corrupted_train_df = train_df.copy()
CORRUPT_COLS = ["revenue", "runtime", "original_language"]
for col in CORRUPT_COLS:
    corrupted_train_df[col] = corrupted_train_df[col].sample(frac=1.0, random_state=config.RANDOM_SEED).to_numpy()
call3_time = timed_transform(corrupted_train_df)

print(f"call 1 (train_df):           {call1_time * 1000:.1f} ms")
print(f"call 2 (train_df):           {call2_time * 1000:.1f} ms")
print(f"call 3 (corrupted train_df): {call3_time * 1000:.1f} ms")

times = [call1_time, call2_time, call3_time]
ratio = max(times) / min(times)
print(f"max/min ratio across the three calls: {ratio:.2f}x")

MAX_ACCEPTABLE_RATIO = 3.0
if ratio > MAX_ACCEPTABLE_RATIO:
    print(
        f"WARNING: {ratio:.2f}x spread exceeds {MAX_ACCEPTABLE_RATIO}x -- something "
        "is doing unnecessary per-call work (e.g. re-deriving the SBERT cache "
        "key per row) and needs fixing before this goes near a SCARF loop calling "
        "transform() twice per batch, every batch"
    )
else:
    print(
        f"All three calls within {MAX_ACCEPTABLE_RATIO}x of each other: no re-fitting "
        "or per-call disk reads, and corruption does not change transform()'s cost"
    )